# YOLOv7 Volleyball Detection

**Pipeline:** Clone Repo → Dataset Download → Training → Validation → Testing → CSV Metrics Export

**Classes:**
- 0: ball
- 1: player
- 2: player 1
- 3: referee

**Note:** YOLOv7 uses the WongKinYiu/yolov7 repository, not ultralytics.

## 1. Setup & Install Dependencies

In [ ]:
!pip install roboflow torch torchvision tqdm matplotlib scipy pyyaml tensorboard seaborn pandas requests

In [ ]:
import os
import sys
import csv
import re
import json
import logging
import subprocess
from datetime import datetime

import torch
from roboflow import Roboflow

print(f"Python      : {sys.version}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

## 2. Clone YOLOv7 Repository

In [ ]:
SCRIPT_DIR = os.path.abspath(os.getcwd())
YOLOV7_DIR = os.path.join(SCRIPT_DIR, "yolov7")

if not os.path.isdir(YOLOV7_DIR):
    print("📥 Cloning YOLOv7 repository…")
    !git clone https://github.com/WongKinYiu/yolov7.git "{YOLOV7_DIR}"
else:
    print(f"✅ YOLOv7 repo already exists at: {YOLOV7_DIR}")

# Install YOLOv7 requirements
req_file = os.path.join(YOLOV7_DIR, "requirements.txt")
if os.path.isfile(req_file):
    !pip install -r "{req_file}" -q
    print("✅ YOLOv7 requirements installed")

print(f"YOLOV7_DIR: {YOLOV7_DIR}")

## 3. Download Dataset from Roboflow

In [ ]:
rf = Roboflow(api_key="MUEDCloFvcri2QqbuoB0")
project = rf.workspace("jeevaraj-s").project("volleyball-hwxp2-wq92v")
version = project.version(2)
dataset = version.download("yolov7")

print(f"\n✅ Dataset downloaded to: {dataset.location}")

## 4. Configure Paths

In [ ]:
# ────────────────────────── CONFIGURATION ──────────────────────────
BASE_DIR    = os.path.dirname(SCRIPT_DIR)
DATASET_DIR = os.path.abspath(dataset.location)
DATA_YAML   = os.path.join(DATASET_DIR, "data.yaml")

# YOLOv7 pretrained weights (tiny for faster training)
WEIGHTS_URL  = "https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7-tiny.pt"
WEIGHTS_PATH = os.path.join(YOLOV7_DIR, "yolov7-tiny.pt")

# Training hyper-params
EPOCHS      = 100
IMG_SIZE    = 640
BATCH_SIZE  = 4

# Output directories
PROJECT_DIR = os.path.join(SCRIPT_DIR, "runs")
LOG_DIR     = os.path.join(SCRIPT_DIR, "logs")
RUN_NAME    = "volleyball_train"

print(f"SCRIPT_DIR  : {SCRIPT_DIR}")
print(f"YOLOV7_DIR  : {YOLOV7_DIR}")
print(f"DATASET_DIR : {DATASET_DIR}")
print(f"DATA_YAML   : {DATA_YAML}")
print(f"WEIGHTS_PATH: {WEIGHTS_PATH}")
print(f"PROJECT_DIR : {PROJECT_DIR}")

In [ ]:
# Download pretrained weights if not present
if not os.path.isfile(WEIGHTS_PATH):
    print(f"📥 Downloading YOLOv7-tiny weights…")
    import urllib.request
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)
    print(f"✅ Weights saved to: {WEIGHTS_PATH}")
else:
    print(f"✅ Weights already exist: {WEIGHTS_PATH}")

In [ ]:
# Update data.yaml with absolute paths
import yaml

with open(DATA_YAML, 'r', encoding='utf-8') as f:
    data_config = yaml.safe_load(f)

# Set absolute paths for train/val/test
data_config['train'] = os.path.join(DATASET_DIR, 'train', 'images')
data_config['val']   = os.path.join(DATASET_DIR, 'valid', 'images')
data_config['test']  = os.path.join(DATASET_DIR, 'test', 'images')

with open(DATA_YAML, 'w', encoding='utf-8') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("✅ Updated data.yaml with absolute paths:")
print(f"   train: {data_config['train']}")
print(f"   val  : {data_config['val']}")
print(f"   test : {data_config['test']}")
print(f"   nc   : {data_config['nc']}")
print(f"   names: {data_config['names']}")

## 5. GPU Check

In [ ]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

if torch.cuda.is_available():
    DEVICE = "0"
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ GPU detected: {name}  ({mem:.1f} GB VRAM)")
else:
    DEVICE = "cpu"
    print("❌ No GPU found — falling back to CPU (training will be slow).")

print(f"Using device: {DEVICE}")

## 6. Setup Logger

In [ ]:
os.makedirs(LOG_DIR, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_FILE  = os.path.join(LOG_DIR, f"training_{timestamp}.log")

logger = logging.getLogger("yolov7_train")
logger.setLevel(logging.INFO)
logger.handlers.clear()

ch = logging.StreamHandler(sys.stdout)
ch.setLevel(logging.INFO)
ch.setFormatter(logging.Formatter("%(message)s"))
logger.addHandler(ch)

fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setLevel(logging.INFO)
fh.setFormatter(logging.Formatter("%(asctime)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
logger.addHandler(fh)

logger.info(f"Log file → {LOG_FILE}")

## 7. Train YOLOv7

In [ ]:
TRAIN_OUTPUT_DIR = os.path.join(PROJECT_DIR, RUN_NAME)

logger.info("=" * 60)
logger.info("  YOLOv7 Volleyball Detection — GPU Training")
logger.info("=" * 60)
logger.info(f"  Weights    : {WEIGHTS_PATH}")
logger.info(f"  Dataset    : {DATA_YAML}")
logger.info(f"  Epochs     : {EPOCHS}")
logger.info(f"  Image size : {IMG_SIZE}")
logger.info(f"  Batch size : {BATCH_SIZE}")
logger.info(f"  Output     : {TRAIN_OUTPUT_DIR}")
logger.info("=" * 60)

train_cmd = [
    sys.executable, os.path.join(YOLOV7_DIR, "train.py"),
    "--workers", "0",
    "--device", DEVICE,
    "--batch-size", str(BATCH_SIZE),
    "--data", DATA_YAML,
    "--img", str(IMG_SIZE), str(IMG_SIZE),
    "--cfg", os.path.join(YOLOV7_DIR, "cfg", "training", "yolov7-tiny.yaml"),
    "--weights", WEIGHTS_PATH,
    "--name", RUN_NAME,
    "--project", PROJECT_DIR,
    "--epochs", str(EPOCHS),
    "--exist-ok",
    "--hyp", os.path.join(YOLOV7_DIR, "data", "hyp.scratch.tiny.yaml"),
]

logger.info("Starting training…\n")
logger.info(f"Command: {' '.join(train_cmd)}\n")

process = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=YOLOV7_DIR,
)

for line in process.stdout:
    line = line.rstrip()
    print(line)
    logger.handlers[1].stream.write(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | {line}\n")
    logger.handlers[1].stream.flush()

process.wait()
print(f"\n{'='*60}")
print(f"Training finished with exit code: {process.returncode}")
print(f"{'='*60}")

In [ ]:
# ── Print Training Summary ──
BEST_WEIGHTS = os.path.join(TRAIN_OUTPUT_DIR, "weights", "best.pt")
LAST_WEIGHTS = os.path.join(TRAIN_OUTPUT_DIR, "weights", "last.pt")

logger.info("\n" + "=" * 60)
logger.info("  TRAINING COMPLETE")
logger.info("=" * 60)
logger.info(f"📁 Results saved to : {TRAIN_OUTPUT_DIR}")
logger.info(f"🏆 Best weights     : {BEST_WEIGHTS} {'✅' if os.path.isfile(BEST_WEIGHTS) else '❌'}")
logger.info(f"   Last weights     : {LAST_WEIGHTS} {'✅' if os.path.isfile(LAST_WEIGHTS) else '❌'}")

# Check for generated charts
logger.info("📊 Auto-saved charts:")
for chart in ["results.png", "confusion_matrix.png", "F1_curve.png",
              "P_curve.png", "R_curve.png", "PR_curve.png"]:
    path = os.path.join(TRAIN_OUTPUT_DIR, chart)
    status = "✅" if os.path.isfile(path) else "—"
    logger.info(f"   {status} {chart}")
logger.info("=" * 60)

## 8. Validate YOLOv7

In [ ]:
VAL_OUTPUT_DIR = os.path.join(SCRIPT_DIR, "runs", "val", "volleyball_val")

if not os.path.isfile(BEST_WEIGHTS):
    print(f"❌ Error: Best weights not found at {BEST_WEIGHTS}")
else:
    print(f"✅ Loading best model: {BEST_WEIGHTS}")
    print(f"🚀 Starting Validation…")

    val_cmd = [
        sys.executable, os.path.join(YOLOV7_DIR, "test.py"),
        "--data", DATA_YAML,
        "--img", str(IMG_SIZE),
        "--batch-size", str(BATCH_SIZE),
        "--weights", BEST_WEIGHTS,
        "--device", DEVICE,
        "--name", "volleyball_val",
        "--project", os.path.join(SCRIPT_DIR, "runs", "val"),
        "--exist-ok",
        "--task", "val",
        "--save-json",
        "--save-txt",
        "--save-conf",
        "--verbose",
    ]

    val_process = subprocess.Popen(
        val_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=YOLOV7_DIR,
    )

    val_output = []
    for line in val_process.stdout:
        line = line.rstrip()
        print(line)
        val_output.append(line)

    val_process.wait()

    # Parse validation metrics from output
    val_p = val_r = val_map50 = val_map95 = 0.0
    for line in val_output:
        # Look for the "all" summary line with metrics
        if "all" in line and not line.strip().startswith("Class"):
            parts = line.split()
            try:
                # Format: Class Images Labels P R mAP@.5 mAP@.5:.95
                nums = [p for p in parts if p.replace('.', '').replace('e-', '').replace('e+', '').isdigit() or 
                        (p.startswith('0.') or p.startswith('1.'))]
                float_vals = []
                for p in parts:
                    try:
                        float_vals.append(float(p))
                    except ValueError:
                        continue
                if len(float_vals) >= 6:
                    val_p     = float_vals[-4]
                    val_r     = float_vals[-3]
                    val_map50 = float_vals[-2]
                    val_map95 = float_vals[-1]
            except Exception:
                pass

    print("\n" + "━" * 50)
    print("📈 YOLOv7 VALIDATION METRICS")
    print("━" * 50)
    print(f"Precision:   {val_p:.4f}")
    print(f"Recall:      {val_r:.4f}")
    print(f"mAP@50:      {val_map50:.4f}")
    print(f"mAP@50-95:   {val_map95:.4f}")
    print("━" * 50)
    print(f"📂 Results saved in: {VAL_OUTPUT_DIR}")

## 9. Test YOLOv7

In [ ]:
TEST_OUTPUT_DIR = os.path.join(SCRIPT_DIR, "runs", "val", "volleyball_test")

if not os.path.isfile(BEST_WEIGHTS):
    print(f"❌ Error: Best weights not found at {BEST_WEIGHTS}")
else:
    print(f"✅ Loading best model: {BEST_WEIGHTS}")
    print(f"🚀 Starting Testing…")

    test_cmd = [
        sys.executable, os.path.join(YOLOV7_DIR, "test.py"),
        "--data", DATA_YAML,
        "--img", str(IMG_SIZE),
        "--batch-size", str(BATCH_SIZE),
        "--weights", BEST_WEIGHTS,
        "--device", DEVICE,
        "--name", "volleyball_test",
        "--project", os.path.join(SCRIPT_DIR, "runs", "val"),
        "--exist-ok",
        "--task", "test",
        "--save-json",
        "--save-txt",
        "--save-conf",
        "--verbose",
    ]

    test_process = subprocess.Popen(
        test_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=YOLOV7_DIR,
    )

    test_output = []
    for line in test_process.stdout:
        line = line.rstrip()
        print(line)
        test_output.append(line)

    test_process.wait()

    # Parse test metrics from output
    test_p = test_r = test_map50 = test_map95 = 0.0
    for line in test_output:
        if "all" in line and not line.strip().startswith("Class"):
            parts = line.split()
            try:
                float_vals = []
                for p in parts:
                    try:
                        float_vals.append(float(p))
                    except ValueError:
                        continue
                if len(float_vals) >= 6:
                    test_p     = float_vals[-4]
                    test_r     = float_vals[-3]
                    test_map50 = float_vals[-2]
                    test_map95 = float_vals[-1]
            except Exception:
                pass

    print("\n" + "━" * 50)
    print("📈 YOLOv7 TEST METRICS")
    print("━" * 50)
    print(f"Precision:   {test_p:.4f}")
    print(f"Recall:      {test_r:.4f}")
    print(f"mAP@50:      {test_map50:.4f}")
    print(f"mAP@50-95:   {test_map95:.4f}")
    print("━" * 50)
    print(f"📂 Results saved in: {TEST_OUTPUT_DIR}")

## 10. Extract Training Metrics from results.txt

In [ ]:
# YOLOv7 saves per-epoch metrics to results.txt
# Columns: epoch, gpu_mem, box, obj, cls, total, labels, img_size,
#          P, R, mAP@.5, mAP@.5:.95, val_box, val_obj, val_cls

results_file = os.path.join(TRAIN_OUTPUT_DIR, "results.txt")
train_p = train_r = train_map50 = train_map95 = 0.0

if os.path.isfile(results_file):
    with open(results_file, 'r') as f:
        lines = f.readlines()
    
    if lines:
        # Get the last epoch's metrics (best training metrics)
        last_line = lines[-1].strip()
        parts = last_line.split()
        try:
            float_vals = [float(p) for p in parts]
            # results.txt columns: epoch, mem, box, obj, cls, total, labels, img_size,
            #                      P(8), R(9), mAP50(10), mAP50-95(11), ...
            if len(float_vals) >= 12:
                train_p     = float_vals[8]
                train_r     = float_vals[9]
                train_map50 = float_vals[10]
                train_map95 = float_vals[11]
        except Exception as e:
            print(f"⚠️ Could not parse results.txt: {e}")

    print("━" * 50)
    print("📈 YOLOv7 TRAINING FINAL METRICS (last epoch)")
    print("━" * 50)
    print(f"Precision:   {train_p:.4f}")
    print(f"Recall:      {train_r:.4f}")
    print(f"mAP@50:      {train_map50:.4f}")
    print(f"mAP@50-95:   {train_map95:.4f}")
    print("━" * 50)
else:
    print(f"❌ results.txt not found at: {results_file}")

## 11. Save All Metrics to CSV

In [ ]:
CSV_PATH = os.path.join(SCRIPT_DIR, "metrics_summary.csv")

rows = [
    ["Phase", "Precision", "Recall", "mAP50", "mAP50-95"],
    ["Train",  f"{train_p:.4f}",   f"{train_r:.4f}",   f"{train_map50:.4f}",  f"{train_map95:.4f}"],
    ["Valid",  f"{val_p:.4f}",     f"{val_r:.4f}",     f"{val_map50:.4f}",    f"{val_map95:.4f}"],
    ["Test",   f"{test_p:.4f}",    f"{test_r:.4f}",    f"{test_map50:.4f}",   f"{test_map95:.4f}"],
]

with open(CSV_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print(f"✅ Metrics saved to: {CSV_PATH}")
print()

# Pretty-print the table
print(f"{'Phase':<8} {'Precision':>10} {'Recall':>10} {'mAP50':>10} {'mAP50-95':>10}")
print("─" * 52)
for row in rows[1:]:
    print(f"{row[0]:<8} {row[1]:>10} {row[2]:>10} {row[3]:>10} {row[4]:>10}")

In [ ]:
# ── Final Summary ──
print("\n" + "=" * 60)
print("  YOLOv7 VOLLEYBALL PIPELINE COMPLETE")
print("=" * 60)
print(f"  📂 Training results   : {TRAIN_OUTPUT_DIR}")
print(f"  📂 Validation results : {VAL_OUTPUT_DIR}")
print(f"  📂 Test results       : {TEST_OUTPUT_DIR}")
print(f"  📊 Metrics CSV        : {CSV_PATH}")
print(f"  📝 Training log       : {LOG_FILE}")
print(f"  🏆 Best weights       : {BEST_WEIGHTS}")
print("=" * 60)